In [ ]:
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
import os
import json, csv, time, random
from datetime import datetime
from http.client import RemoteDisconnected
import httplib2

httplib2.Http.timeout = 60

MANUAL_API_KEY = 'YOUR_YOUTUBE_API_KEY_HERE'  
API_KEY = MANUAL_API_KEY.strip()


def build_youtube_client():
    return build(
        'youtube',
        'v3',
        developerKey=API_KEY,
        http=httplib2.Http(timeout=60),
        cache_discovery=False,
    )

USE_BROAD_SEARCH = True

MAX_RETRIES     = 5
INITIAL_BACKOFF = 2
REQUEST_DELAY   = 0.5

REGION_CODE        = 'TW'
MAX_PER_PAGE       = 50
PAGES_PER_CATEGORY = 20

CATEGORY_IDS = {
    '1':  'Film & Animation',
    '2':  'Autos & Vehicles',
    '10': 'Music',
    '15': 'Pets & Animals',
    '17': 'Sports',
    '19': 'Travel & Events',
    '20': 'Gaming',
    '22': 'People & Blogs',
    '23': 'Comedy',
    '24': 'Entertainment',
    '25': 'News & Politics',
    '26': 'Howto & Style',
    '27': 'Education',
    '28': 'Science & Technology',
}

Utility functions

In [63]:
def retry_with_backoff(func):
    for attempt in range(MAX_RETRIES):
        try:
            time.sleep(REQUEST_DELAY)
            return func()
        except (RemoteDisconnected, TimeoutError, ConnectionResetError, BrokenPipeError) as e:
            if attempt < MAX_RETRIES - 1:
                wait = INITIAL_BACKOFF * (2 ** attempt) + random.uniform(0, 1)
                print(f'  Connection error: {e}, retrying in {wait:.1f}s ({attempt+1}/{MAX_RETRIES})')
                time.sleep(wait)
            else:
                raise
        except HttpError as e:
            if e.resp.status in [429, 500, 503]:
                if attempt < MAX_RETRIES - 1:
                    wait = INITIAL_BACKOFF * (2 ** attempt) + random.uniform(0, 1)
                    print(f'  HTTP {e.resp.status}, retrying in {wait:.1f}s ({attempt+1}/{MAX_RETRIES})')
                    time.sleep(wait)
                else:
                    raise
            else:
                raise

Search videos

In [64]:
def search_by_category(category_id, pages=PAGES_PER_CATEGORY):
    """Fetch trending videos by category ID, return a list of video IDs"""
    youtube = build_youtube_client()
    all_videos = []
    next_page_token = None

    for page in range(pages):
        print(f'  Page {page+1}/{pages}...', end='', flush=True)

        def _fetch():
            return youtube.videos().list(
                part='snippet',
                chart='mostPopular',
                regionCode=REGION_CODE,
                videoCategoryId=category_id,
                maxResults=MAX_PER_PAGE,
                pageToken=next_page_token
            ).execute()

        response = retry_with_backoff(_fetch)
        items = response.get('items', [])
        print(f' {len(items)} videos')

        for item in items:
            all_videos.append({
                'video_id': item['id'],
                'channel_id': item['snippet']['channelId'],
            })

        next_page_token = response.get('nextPageToken')
        if not next_page_token:
            print('  No more pages')
            break

        time.sleep(1.0)

    return all_videos


def search_broadly(pages=PAGES_PER_CATEGORY, query='台灣'):
    """Broad search across all categories, prioritizing Taiwan region + Traditional Chinese"""
    youtube = build_youtube_client()
    all_videos = []
    next_page_token = None

    for page in range(pages):
        print(f'  Page {page+1}/{pages}...', end='', flush=True)

        def _fetch_strict():
            return youtube.search().list(
                part='snippet',
                q=query,
                maxResults=MAX_PER_PAGE,
                type='video',
                regionCode='TW',
                relevanceLanguage='zh-Hant',
                order='viewCount',
                videoDefinition='high',
                pageToken=next_page_token
            ).execute()

        response = retry_with_backoff(_fetch_strict)
        items = response.get('items', [])

        # If the first page returns no data at all, relax the filters and retry
        if page == 0 and not items:
            def _fetch_relaxed():
                return youtube.search().list(
                    part='snippet',
                    q=query,
                    maxResults=MAX_PER_PAGE,
                    type='video',
                    order='viewCount',
                    videoDefinition='high',
                    pageToken=next_page_token
                ).execute()
            response = retry_with_backoff(_fetch_relaxed)
            items = response.get('items', [])
            print(f' {len(items)} videos (filters relaxed)')
        else:
            print(f' {len(items)} videos')

        for item in items:
            video_id = item.get('id', {}).get('videoId')
            channel_id = item.get('snippet', {}).get('channelId')
            if not video_id or not channel_id:
                continue
            all_videos.append({
                'video_id': video_id,
                'channel_id': channel_id,
            })

        next_page_token = response.get('nextPageToken')
        if not next_page_token:
            print('  No more pages')
            break

        time.sleep(1.0)

    return all_videos

Raw video data

In [65]:
def get_video_raw(video_id):
    """Fetch raw video data (no computation applied)"""
    youtube = build_youtube_client()

    def _fetch():
        return youtube.videos().list(
            part='snippet,statistics,contentDetails,topicDetails',
            id=video_id
        ).execute()

    try:
        response = retry_with_backoff(_fetch)
        if not (response and response['items']):
            return None

        v       = response['items'][0]
        stats   = v.get('statistics', {})
        content = v.get('contentDetails', {})
        topics  = v.get('topicDetails', {})

        return {
            'video_id':          video_id,
            'title':             v['snippet']['title'],
            'description':       v['snippet'].get('description') or None,
            'channel_id':        v['snippet']['channelId'],
            'channel_name':      v['snippet']['channelTitle'],
            'published_at':      v['snippet']['publishedAt'],
            'view_count':        int(stats.get('viewCount', 0)),
            'like_count':        int(stats.get('likeCount', 0)),
            'comment_count':     int(stats.get('commentCount', 0)),
            'duration':          content.get('duration', 'PT0S'),
            'topic_categories':  topics.get('topicCategories', []),  # raw Wikipedia URLs
        }
    except Exception as e:
        print(f'  Video failed {video_id}: {e}')
        return None

Raw channel data

In [66]:
def get_channel_raw(channel_id):
    """Fetch raw channel data (no computation applied)"""
    youtube = build_youtube_client()

    def _fetch():
        return youtube.channels().list(
            part='snippet,statistics,topicDetails',
            id=channel_id
        ).execute()

    try:
        response = retry_with_backoff(_fetch)
        if not response['items']:
            return None

        ch    = response['items'][0]
        stats = ch.get('statistics', {})
        snip  = ch['snippet']

        return {
            'channel_id':        channel_id,
            'channel_name':      snip['title'],
            'custom_url':        snip.get('customUrl', None),
            'country':           snip.get('country', None),
            'created_at':        snip['publishedAt'],
            'subscriber_count':  int(stats.get('subscriberCount', 0)),
            'view_count':        int(stats.get('viewCount', 0)),
            'video_count':       int(stats.get('videoCount', 0)),
            'topic_categories':  ch.get('topicDetails', {}).get('topicCategories', []),
        }
    except Exception as e:
        print(f'  Channel failed {channel_id}: {e}')
        return None

Save functions

In [67]:
def save_json(data, filename):
    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(data, f, indent=2, ensure_ascii=False)
    print(f'Saved: {filename} ({len(data)} records)')


def save_csv(data, filename):
    if not data:
        return
    rows = []
    for item in data:
        row = {
            k: (json.dumps(v, ensure_ascii=False) if isinstance(v, list) else v)
            for k, v in item.items()
        }
        rows.append(row)

    fieldnames = list(rows[0].keys())
    with open(filename, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)
    print(f'Saved: {filename} ({len(rows)} records)')

Main Code

In [ ]:
if __name__ == '__main__':
    timestamp     = datetime.now().strftime('%Y%m%d_%H%M%S')
    all_videos    = []   
    all_channels  = {}  

    if USE_BROAD_SEARCH:
        print('\n[Broad search mode] Taiwan region + Traditional Chinese')
        print('=' * 60)
        raw_list = search_broadly()
        print(f'Retrieved {len(raw_list)} video IDs')

        for i, item in enumerate(raw_list, 1):
            vid = item['video_id']
            print(f'  [{i}/{len(raw_list)}] video_id: {vid}')

            video_data = get_video_raw(vid)
            if not video_data:
                continue

            video_data['category_id']   = None
            video_data['category_name'] = 'Broad Search'
            all_videos.append(video_data)

            ch_id = video_data['channel_id']
            if ch_id not in all_channels:
                print(f'    → Fetching channel: {video_data["channel_name"]}')
                ch_data = get_channel_raw(ch_id)
                if ch_data:
                    all_channels[ch_id] = ch_data

            time.sleep(1.5)
    else:
        for cat_id, cat_name in CATEGORY_IDS.items():
            print(f'\n[Category {cat_id}] {cat_name}')
            print('=' * 60)

            raw_list = search_by_category(cat_id)
            print(f'Retrieved {len(raw_list)} video IDs')

            for i, item in enumerate(raw_list, 1):
                vid = item['video_id']
                print(f'  [{i}/{len(raw_list)}] video_id: {vid}')

                video_data = get_video_raw(vid)
                if not video_data:
                    continue

                video_data['category_id']   = cat_id
                video_data['category_name'] = cat_name
                all_videos.append(video_data)

                ch_id = video_data['channel_id']
                if ch_id not in all_channels:
                    print(f'    → Fetching channel: {video_data["channel_name"]}')
                    ch_data = get_channel_raw(ch_id)
                    if ch_data:
                        all_channels[ch_id] = ch_data

                time.sleep(1.5)

    print(f'\nDone! {len(all_videos)} videos, {len(all_channels)} channels')

    channel_map = {ch['channel_id']: ch for ch in all_channels.values() if ch.get('channel_id')}
    merged_records = []
    for v in all_videos:
        ch = channel_map.get(v.get('channel_id'), {})
        merged = dict(v)
        merged.update({
            'channel_custom_url': ch.get('custom_url'),
            'channel_country': ch.get('country'),
            'channel_created_at': ch.get('created_at'),
            'channel_subscriber_count': ch.get('subscriber_count'),
            'channel_view_count': ch.get('view_count'),
            'channel_video_count': ch.get('video_count'),
            'channel_topic_categories': ch.get('topic_categories', []),
        })
        merged_records.append(merged)

    merged_file = f'merged_final_brands_{timestamp}.json'
    save_json(merged_records, merged_file)
    print(f'Merge complete: {merged_file} ({len(merged_records)} records)')